
# Tutorial: **Limpieza de Datos** (Pandas)

**Objetivo**: Aplicar técnicas prácticas de **limpieza** sobre un dataset con problemas reales: valores faltantes, duplicados, tipos mezclados, formatos inconsistentes, outliers y categorías no estandarizadas.



<a id="material"></a>
## 1) Material de estudio



<a id="carga"></a>
### 1.1 Cargar datos “sucios” (CSV/Excel)

La carpeta **`datasets_limpieza`** tiene los archivos para este tutorial.

Archivos provistos:
- `clientes_sucios.csv`
- `productos_sucios.csv`
- `ventas_sucias.csv`
- `datos_sucios.xlsx` (hojas: `clientes`, `ventas`)


In [1]:
import pandas as pd

import numpy as np


In [2]:


clientes_path = "datasets_limpieza/clientes_sucios.csv"
productos_path = "datasets_limpieza/productos_sucios.csv"
ventas_path    = "datasets_limpieza/ventas_sucias.csv"
excel_path     = "datasets_limpieza/datos_sucios.xlsx"

print("\nLeyendo CSVs principales...")
clientes  = pd.read_csv(clientes_path)
productos = pd.read_csv(productos_path)
ventas    = pd.read_csv(ventas_path)

print("Filas/Columnas -> clientes:", clientes.shape, "| productos:", productos.shape, "| ventas:", ventas.shape)

print("\nLeyendo Excel de referencia (primeras filas)...")
xls_clientes = pd.read_excel(excel_path, sheet_name="clientes")
xls_ventas   = pd.read_excel(excel_path, sheet_name="ventas")
print("Excel clientes:", xls_clientes.shape, "| Excel ventas:", xls_ventas.shape)


Leyendo CSVs principales...
Filas/Columnas -> clientes: (61, 8) | productos: (12, 5) | ventas: (200, 9)

Leyendo Excel de referencia (primeras filas)...
Excel clientes: (50, 8) | Excel ventas: (100, 9)



<a id="diagnostico"></a>
### 1.2 Diagnóstico rápido

Análisis Exploratorio de Datos (EDA)

- Vista rápida: `head()`, `sample()`  
- Esquema y tipos: `info()`, `dtypes`, `columns`, `shape`  
- Calidad de datos: `isna().sum()`, `duplicated().sum()`  
- Cardinalidad: `nunique()`  
- Reglas simples (ej.: `cantidad > 0`)


In [3]:

print("Primeras filas de `ventas`:")
print(ventas.head())

print("\nResumen de tipos en `clientes`:")
print(clientes.dtypes)


Primeras filas de `ventas`:
   id_venta cliente_id  producto_id       fecha cantidad precio_unitario  \
0     50000       1054         2007  05/09/2025        1            $-10   
1     50001       1049         2001  02/02/2025        2         $19.990   
2     50002     001045         2099    23-07-24        2             NaN   
3     50003       1032         2001    13-01-25      dos             NaN   
4     50004       1032         2099    05-05-24        1        159.990    

  descuento metodo_pago total_venta  
0       5 %         TDC          $0  
1      0,15     Crédito     999.999  
2       5 %     Crédito         NaN  
3       10%         TDC         NaN  
4      0,15    Efectivo         NaN  

Resumen de tipos en `clientes`:
id_cliente          int64
nombre             object
email              object
ciudad             object
fecha_registro     object
genero             object
edad              float64
telefono           object
dtype: object


In [4]:

print("\nValores faltantes por columna (productos):")
print(productos.isna().sum())

print("\nDuplicados totales en clientes:", clientes.duplicated().sum())
print("Valores únicos por columna en metodo_pago (ventas):")
print(ventas["metodo_pago"].value_counts(dropna=False))


Valores faltantes por columna (productos):
id_producto    0
producto       0
categoria      0
precio         0
activo         0
dtype: int64

Duplicados totales en clientes: 1
Valores únicos por columna en metodo_pago (ventas):
Crédito     33
Débito      30
tarjeta     28
Debito      26
efec.       25
TDC         23
Efectivo    20
Tarjeta     15
Name: metodo_pago, dtype: int64



<a id="tipificacion"></a>
### 1.3 Tipificación y normalización

- **Fechas**: `pd.to_datetime(..., errors="coerce", dayfirst=True)`  
- **Números en texto**: quitar símbolos y separadores, `pd.to_numeric(..., errors="coerce")`  
- **Texto**: `.str.strip()`, `.str.lower()`
- **Categorización**: mapear sinónimos/variaciones (ej.: método de pago, género)


In [5]:

import numpy as np
import re

def parse_moneda(s):
    """Convierte strings como '$19.990', '29,990', ' 159.990 ' a float (CLP).
    Retorna NaN si no es posible."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    s = s.replace("$","").replace(" ","")
    # Cambiar coma por punto SOLO si hay coma decimal
    s = s.replace(".", "")  # quita separador de miles
    s = s.replace(",", ".") # coma decimal a punto
    try:
        return float(s)
    except:
        return np.nan




# Aplicar en copias para no perder el original
ventas_fix = ventas.copy()
productos_fix = productos.copy()
clientes_fix = clientes.copy()

# Fechas
ventas_fix["fecha"] = pd.to_datetime(ventas_fix["fecha"], errors="coerce", dayfirst=True)

# Números: cantidad y precio_unitario
## Aplicamos funcion parse_moneda
ventas_fix["cantidad"] = pd.to_numeric(ventas_fix["cantidad"].astype(str).str.strip(), errors="coerce")
ventas_fix["precio_unitario"] = ventas_fix["precio_unitario"].apply(parse_moneda)


In [6]:

# Descuento: '10%' -> 0.10 ; '0,15' -> 0.15 ; None -> 0
def parse_desc(s):
    if pd.isna(s) or str(s).strip() == "":
        return 0.0
    st = str(s).strip().replace(" ", "")
    st = st.replace(",", ".")
    if st.endswith("%"):
        try:
            return float(st[:-1]) / 100.0
        except:
            return np.nan
    try:
        val = float(st)
        return val if val <= 1 else val/100.0
    except:
        return np.nan

ventas_fix["descuento"] = ventas_fix["descuento"].apply(parse_desc)


In [7]:

def normaliza_texto(x, modo="title"):
    if pd.isna(x):
        return x
    s = str(x).strip()
    if modo == "lower":
        return s.lower()
    if modo == "upper":
        return s.upper()
    return s.title()

# Normalizar categorías comunes
metodo_map = {
    "tarjeta":"Tarjeta", "tdc":"Tarjeta", "crédito":"Tarjeta", "credito":"Tarjeta", "tarj.":"Tarjeta",
    "efectivo":"Efectivo", "efec.":"Efectivo",
    "debito":"Débito", "débito":"Débito"
}



# Texto/categorías
ventas_fix["metodo_pago"] = ventas_fix["metodo_pago"].astype(str).str.strip().str.lower().map(metodo_map).fillna(ventas_fix["metodo_pago"])
clientes_fix["ciudad"] = clientes_fix["ciudad"].apply(lambda s: normaliza_texto(s, modo="title"))
clientes_fix["nombre"] = clientes_fix["nombre"].apply(lambda s: normaliza_texto(s, modo="title"))

productos_fix["categoria"] = productos_fix["categoria"].apply(lambda s: normaliza_texto(s, modo="title").replace("Audio Video","Audio/Video").replace("Computacion","Computación"))
productos_fix["precio"] = productos_fix["precio"].apply(parse_moneda)

# Ids a numéricos (coerce) y luego a Int64 (nullable)
clientes_fix["id_cliente_num"] = pd.to_numeric(clientes_fix["id_cliente"], errors="coerce").astype("Int64")
ventas_fix["cliente_id_num"] = pd.to_numeric(ventas_fix["cliente_id"], errors="coerce").astype("Int64")
ventas_fix["producto_id_num"] = pd.to_numeric(ventas_fix["producto_id"], errors="coerce").astype("Int64")

# Edad a numérica
clientes_fix["edad"] = pd.to_numeric(clientes_fix["edad"], errors="coerce")


<a id="limpieza"></a>
### 1.4 Limpieza paso a paso

Ejemplos prácticos:
- **Filtrar registros imposibles** (p. ej., `cantidad <= 0`).
- **Imputar** faltantes simples (mediana/moda) o dejar `NaN` documentado.
- **Eliminar duplicados** (total o por claves relevantes).
- **Recalcular variables derivadas** (por ej., `total_venta`).

In [8]:
# 1) Filtrar imposibles en ventas: cantidad <= 0 y precio_unitario < 0
mask_imposibles = (ventas_fix["cantidad"].isna()) | (ventas_fix["cantidad"] <= 0) | (ventas_fix["precio_unitario"] < 0)
print("Registros imposibles en ventas (cantidad<=0 o precio<0):", int(mask_imposibles.sum()))
ventas_fix = ventas_fix.loc[~mask_imposibles].copy()


Registros imposibles en ventas (cantidad<=0 o precio<0): 85


In [9]:
# 2) Imputación simple en clientes: edad outliers -> NaN, luego imputar mediana
q1, q3 = clientes_fix["edad"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = (clientes_fix["edad"] < lim_inf) | (clientes_fix["edad"] > lim_sup)
print("Outliers de edad detectados:", int(outliers.sum()))
clientes_fix.loc[outliers, "edad"] = np.nan

mediana_edad = clientes_fix["edad"].median(skipna=True)
clientes_fix["edad"] = clientes_fix["edad"].fillna(mediana_edad)
print("Mediana de edad utilizada para imputación:", mediana_edad)

Outliers de edad detectados: 0
Mediana de edad utilizada para imputación: 29.0


In [10]:
# 3) Duplicados: eliminar duplicados exactos de filas en clientes
dups_total = clientes_fix.duplicated().sum()
print("Duplicados exactos en clientes:", int(dups_total))
clientes_fix = clientes_fix.drop_duplicates().copy()


Duplicados exactos en clientes: 1


In [11]:
# 4) Duplicados por clave lógica (email)
dups_email = clientes_fix.duplicated(subset=["email"], keep=False).sum()
print("Registros con email duplicado (conteo de filas):", int(dups_email))


Registros con email duplicado (conteo de filas): 19


In [12]:
# 5) Recalcular total_venta con descuento
ventas_fix["total_calc"] = ventas_fix["cantidad"] * ventas_fix["precio_unitario"] * (1 - ventas_fix["descuento"].fillna(0))
print("\nMuestra de cálculo total_venta:")
print(ventas_fix[["cantidad","precio_unitario","descuento","total_calc"]].head())


Muestra de cálculo total_venta:
    cantidad  precio_unitario  descuento  total_calc
1        2.0          19990.0       0.15     33983.0
2        2.0              NaN       0.05         NaN
4        1.0         159990.0       0.15    135991.5
11       2.0              0.0       0.10         0.0
12       5.0         159990.0       0.05    759952.5



<a id="validacion"></a>
### 1.5 Validación post-limpieza

- Verifica `NaN` remanentes y tamaños finales.  
- Comprueba reglas sencillas (fechas válidas, rangos numéricos, categorías esperadas).

In [13]:

print("NaNs restantes por columna en ventas_fix:")
print(ventas_fix.isna().sum())

print("\nRangos de cantidad en ventas:", (ventas_fix["cantidad"].min(), ventas_fix["cantidad"].max()))
print("Fechas válidas en ventas (NaT):", ventas_fix["fecha"].isna().sum(), "registros con fecha no válidas")

NaNs restantes por columna en ventas_fix:
id_venta            0
cliente_id          0
producto_id         0
fecha               0
cantidad            0
precio_unitario    15
descuento           0
metodo_pago         0
total_venta        53
cliente_id_num      0
producto_id_num     0
total_calc         15
dtype: int64

Rangos de cantidad en ventas: (1.0, 5.0)
Fechas válidas en ventas (NaT): 0 registros con fecha no válidas



<a id="guardar"></a>
### 1.6 Guardar datasets limpios

Guardamos versiones limpias listas para análisis/joins posteriores.


In [14]:
clientes_out = clientes_fix.copy()
ventas_out = ventas_fix.copy()
productos_out = productos_fix.copy()

clientes_out.to_csv("datasets_limpieza/clientes_limpios.csv", index=False)
ventas_out.to_csv("datasets_limpieza/ventas_limpias.csv", index=False)
productos_out.to_csv("datasets_limpieza/productos_limpios.csv", index=False)

with pd.ExcelWriter("datasets_limpieza/datos_limpios.xlsx") as writer:
    clientes_out.to_excel(writer, index=False, sheet_name="clientes")
    ventas_out.to_excel(writer, index=False, sheet_name="ventas")
    productos_out.to_excel(writer, index=False, sheet_name="productos")



<a id="ejercicios"></a>
## 2) Ejercicios

> **Tip**: usa impresiones descriptivas en cada paso (por ejemplo: `print("Duplicados por email:", df.duplicated('email').sum())`).



### Ejercicio 1 — Diagnóstico
1. Reporta filas/columnas y `dtypes` de **clientes**, **productos** y **ventas**.  
2. Muestra `isna().sum()` y `duplicated().sum()` de cada DataFrame.  
3. Lista los valores únicos de `metodo_pago` y normalízalos a un conjunto consistente.




### Ejercicio 2 — Tipos y parsing
1. Convierte `fecha` de **ventas** a `datetime` con `dayfirst=True`.  
2. Normaliza `precio_unitario` y `precio` (quita símbolos de moneda y separadores).  
3. Convierte `cantidad` y `edad` a valores numéricos; documenta filas no válidas.



### Ejercicio 3 — Reglas de negocio
1. Filtra ventas con `cantidad <= 0` o `precio_unitario < 0`, también filtra iltra ventas con `cantidad <= 0` y `precio_unitario < 0. ¿Da el mismo resultado? 
2. Recalcula `total_venta` como `cantidad * precio_unitario * (1 - descuento)`, aplica un aumento solo a los con precio unitario mas bajo que el promedio.  
3. ¿Cuántos registros tenían `total_venta` inconsistente/ausente? Reporta el porcentaje.



### Ejercicio 4 — Duplicados 
1. Elimina duplicados totales en **clientes** y duplicados por clave `email`. ¿Cuántos eliminaste?
2. Elimina los clientes que su mail contiene `@ejemplo`. ¿Cuántos son?


### Ejercicio 5 — Normalización de texto
1. Homogeneiza `ciudad` en **clientes** y `categoria` en **productos** (capitalización).  
2. Estandariza `genero` a `{"F","M"}` como se hizo antes (o parecido a) `metodo_pago` a `{"Tarjeta","Débito","Efectivo"}`.  
3. (Opcional) Normaliza teléfonos dejando solo dígitos y el largo esperado.



### Ejercicio 6 — Guardado y verificación
1. Exporta **clientes_limpios**, **ventas_limpias**, **productos_limpios** a CSV y Excel. Utiliza index = True, ¿Cuál es la diferencia? 
2. Vuelve a leerlos y valida: filas/columnas, `dtypes`, ausencia de duplicados y reglas básicas. Igual que antes